# PKG — Account Ego Network Explorer

Enter a **PNC deposit account number** and a date window. The app builds the account's
ego network from the payment staging table (`neo4j_payments`) and shows:

* **Network** — the account (ego), every account that paid it or was paid by it (alters),
  and the payments *between* those alters. Direction is drawn as arrows.
  PNC customer accounts and external counterparty accounts are drawn in different colours and shapes.
* **Summary / Senders & receivers / Rails / Over time / External banks / Ties** — tables and
  figures describing both ends of the account's flow.

**How a row is read.** One row per transaction. A side is **internal** (PNC customer) when it
carries an MDM id or a PNC deposit account; otherwise it is **external**, identified by
`unq_cpty_acct_id`. Nodes are **accounts**, not customers: a customer with three accounts is
three nodes, and accounts that belong to the ego's own customer are flagged as *same customer*.

**What cannot be seen.** External → external payments are not in the table, so a tie between two
external counterparties is never observable. Every alter–alter tie shown has at least one PNC end.

Run all cells, then use the app at the bottom. Everything the app shows is also written to
`OUT_ROOT/<account>_<start>_<end>/` (CSV tables, PNG figures, the network as standalone HTML).

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
PAYMENTS_TABLE  = "neo4j_payments"   # qualify with its database: "<db>.neo4j_payments"

DEFAULT_START   = "2024-01-01"       # pre-2024 data is excluded as unreliable
DEFAULT_END     = "2025-12-31"

OUT_ROOT        = "../ego"           # one sub-folder per (account, window)

# Alter<->alter ties need a second pass over the whole window. The pass is restricted to the
# top-ALTER_CAP alters by dollar flow with the ego: a hub account can have 10^5 alters, and ties
# among the long tail are neither drawable nor readable. Raise it if you need the full picture.
ALTER_CAP       = 500
MAX_VIS_NODES   = 150      # alters drawn (top by $ flow with ego). Tables always cover every alter.
TOP_N           = 25       # rows in the top-sender / top-receiver tables
DIST_SAMPLE_MAX = 300_000  # ego transactions pulled to pandas for the amount histogram
SHOW_FIGURES    = True     # figures are always saved to disk; this also renders them in the app

In [ ]:
import os, re, json, time, html as _html
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, HTML
import ipywidgets as W
from pyvis.network import Network

from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("pkg_ego_network").enableHiveSupport().getOrCreate()

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 250)

# ---- visual vocabulary (shared by the network, figures and tables) ---------------------------
C_EGO, C_INT, C_EXT, C_SIB = "#F58025", "#1F5AA6", "#2E9E6B", "#F9C89B"
TYPE_LABEL = {"internal": "PNC customer", "external": "External counterparty"}
TYPE_COLOR = {"internal": C_INT, "external": C_EXT}
DIR_LABEL  = {"IN": "Received by account", "OUT": "Sent by account"}

_RAIL_FIXED = [("ACH", "#1F77B4"), ("WIRE", "#D62728"), ("RTP", "#9467BD"), ("R2P", "#9467BD"),
               ("CHECK", "#8C564B"), ("CHEQUE", "#8C564B"), ("DEBIT", "#E377C2"),
               ("CARD", "#E377C2"), ("BOOK", "#7F7F7F"), ("UNKNOWN", "#C7C7C7")]
_RAIL_EXTRA = ["#17BECF", "#BCBD22", "#FF7F0E", "#AEC7E8", "#98DF8A", "#C5B0D5", "#FFBB78", "#9EDAE5"]
_RAIL_CACHE = {}

def rail_color(rail):
    """Stable colour per rail across every figure and the network. Matched on substring so
    'ACH' and 'ACH_CCD' share a hue family; unseen rails take the next spare colour."""
    r = str(rail).upper()
    if r not in _RAIL_CACHE:
        hit = next((c for k, c in _RAIL_FIXED if k in r), None)
        if hit is None:
            used = {v for v in _RAIL_CACHE.values()}
            hit = next((c for c in _RAIL_EXTRA if c not in used), "#999999")
        _RAIL_CACHE[r] = hit
    return _RAIL_CACHE[r]

def _money_short(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    a, s = abs(x), "-" if x < 0 else ""
    for d, u in ((1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{s}${a / d:,.1f}{u}"
    return f"{s}${a:,.0f}"

def _trunc(s, n=28):
    s = "" if s is None or (isinstance(s, float) and np.isnan(s)) else str(s)
    return s if len(s) <= n else s[: n - 1] + "…"

def _log(msg, t0=None, sink=print):
    sink(f"[{time.strftime('%H:%M:%S')}] {msg}" + (f"  ({time.time() - t0:.1f}s)" if t0 else ""))

## 1 · Data layer (PySpark)

`standardize()` turns each staging row into `source → dest` form with a typed key per side:
`PNC:<deposit account>` for internal, `EXT:<unq_cpty_acct_id>` for external. The same function
is used for the ego pass and for the alter–alter pass, so the two can never disagree on identity.

In [ ]:
REQUIRED = ["trans_id", "trans_dt", "trans_amt",
            "mdm_id_pays", "customer_name_pays", "mdm_id_receives", "customer_name_receives",
            "pnc_dep_acct_pays", "pnc_dep_acct_receives", "unq_cpty_acct_id",
            "cpty_name", "cpty_fin_entity_name", "payment_rail", "category"]
ID_COLS = ["mdm_id_pays", "mdm_id_receives", "pnc_dep_acct_pays", "pnc_dep_acct_receives",
           "unq_cpty_acct_id"]


def load_payments(start, end, sink=print):
    """Window-filtered projection of the staging table. Nothing is materialised here."""
    df = spark.table(PAYMENTS_TABLE)
    missing = sorted(set(REQUIRED) - set(df.columns))
    if missing:
        raise ValueError(f"{PAYMENTS_TABLE} is missing required columns: {missing}")
    types = dict(df.dtypes)
    non_str = {c: types[c] for c in ID_COLS if types[c] != "string"}
    if non_str:
        # Numeric ids silently lose leading zeros; an account typed '000123' would then never match.
        sink(f"WARNING: id columns are not STRING {non_str}. They are cast to string, but any "
             f"leading zeros are already gone. Enter the account number without them.")
    # Upper bound is exclusive on the next day, so it is correct whether trans_dt is DATE,
    # TIMESTAMP or a 'YYYY-MM-DD' string (lexical order == date order), and it keeps partition
    # pruning if the table is partitioned on trans_dt.
    end_excl = (pd.Timestamp(end) + timedelta(days=1)).strftime("%Y-%m-%d")
    cols = [F.col(c).cast("string").alias(c) if c in ID_COLS else F.col(c) for c in REQUIRED]
    return (df.where((F.col("trans_dt") >= F.lit(start)) & (F.col("trans_dt") < F.lit(end_excl)))
              .select(*cols))


def _txt(c):
    """Trimmed text, with '' normalised to NULL (one representation of 'absent')."""
    s = F.trim(F.col(c))
    return F.when(F.length(s) > 0, s)


def _id(c):
    """Ids are compared exactly (no trim) but '' is treated as absent."""
    return F.when(F.length(F.col(c)) > 0, F.col(c))


def standardize(df):
    s_acct, d_acct = _id("pnc_dep_acct_pays"), _id("pnc_dep_acct_receives")
    s_mdm, d_mdm = _id("mdm_id_pays"), _id("mdm_id_receives")
    cpty = _id("unq_cpty_acct_id")
    # A side is internal if it has an MDM id (the documented direction rule) OR a PNC deposit
    # account. The account test is needed because the ego is selected on the account column:
    # a row carrying the ego's account with a null MDM would otherwise type the ego as external
    # and break the ego key. Rows where the two tests disagree are counted in QA.
    s_int = s_mdm.isNotNull() | s_acct.isNotNull()
    d_int = d_mdm.isNotNull() | d_acct.isNotNull()

    def key(is_int, acct, mdm):
        # An internal side with no account falls back to its MDM id so it still has an identity.
        # External side with no counterparty id -> NULL key -> counted and left out of the network.
        return (F.when(is_int, F.concat(F.lit("PNC:"), F.coalesce(acct, F.concat(F.lit("MDM-"), mdm))))
                 .otherwise(F.concat(F.lit("EXT:"), cpty)))

    return (df.select(
        F.col("trans_id").cast("string").alias("trans_id"),
        F.to_date(F.col("trans_dt")).alias("dt"),
        F.col("trans_amt").cast("double").alias("amt"),
        F.coalesce(F.upper(_txt("payment_rail")), F.lit("UNKNOWN")).alias("rail"),
        F.coalesce(F.lower(_txt("category")), F.lit("unknown")).alias("category"),
        F.when(s_int, "internal").otherwise("external").alias("src_type"),
        F.when(d_int, "internal").otherwise("external").alias("dst_type"),
        key(s_int, s_acct, s_mdm).alias("src_key"),
        key(d_int, d_acct, d_mdm).alias("dst_key"),
        F.when(s_int, s_acct).otherwise(cpty).alias("src_acct"),
        F.when(d_int, d_acct).otherwise(cpty).alias("dst_acct"),
        s_mdm.alias("src_mdm"), d_mdm.alias("dst_mdm"),
        F.when(s_int, _txt("customer_name_pays")).otherwise(_txt("cpty_name")).alias("src_name"),
        F.when(d_int, _txt("customer_name_receives")).otherwise(_txt("cpty_name")).alias("dst_name"),
        _txt("cpty_fin_entity_name").alias("cpty_bank"),
        ((s_acct.isNotNull() & s_mdm.isNull()).cast("int")
         + (d_acct.isNotNull() & d_mdm.isNull()).cast("int")).alias("qa_acct_without_mdm"),
    ).withColumn("month", F.date_format("dt", "yyyy-MM")))


def ego_transactions(pay, acct):
    """Every transaction touching the account, oriented from the ego's point of view."""
    ego_key = f"PNC:{acct}"
    raw = pay.where((F.col("pnc_dep_acct_pays") == acct) | (F.col("pnc_dep_acct_receives") == acct))
    e = standardize(raw)
    is_out, is_in = F.col("src_key") == ego_key, F.col("dst_key") == ego_key
    e = e.withColumn("direction", F.when(is_out & is_in, "SELF").when(is_out, "OUT")
                                   .when(is_in, "IN").otherwise("ANOMALY"))
    on_in = F.col("direction") == "IN"

    def cp(src_col, dst_col):   # the other end of the transaction
        return F.when(on_in, F.col(src_col)).otherwise(F.col(dst_col))

    def me(src_col, dst_col):   # the ego's own end
        return F.when(on_in, F.col(dst_col)).otherwise(F.col(src_col))

    return (e.withColumn("cp_key", cp("src_key", "dst_key"))
             .withColumn("cp_type", cp("src_type", "dst_type"))
             .withColumn("cp_acct", cp("src_acct", "dst_acct"))
             .withColumn("cp_mdm", cp("src_mdm", "dst_mdm"))
             .withColumn("cp_name", cp("src_name", "dst_name"))
             # cpty_fin_entity_name describes the counterparty's bank, so it belongs to the
             # external end only; an internal end banks at PNC by definition.
             .withColumn("cp_bank", F.when(F.col("cp_type") == "external", F.col("cpty_bank")))
             .withColumn("ego_mdm", me("src_mdm", "dst_mdm"))
             .withColumn("ego_name", me("src_name", "dst_name"))), ego_key


def diagnose_missing(pay, acct):
    """Zero rows usually means a formatting mismatch, not an unused account. Look for the id with
    whitespace and leading zeros stripped, on both PNC columns and the counterparty column."""
    target = acct.strip().lstrip("0")
    norm = lambda c: F.regexp_replace(F.trim(F.col(c)), "^0+", "")
    hits = []
    for c in ["pnc_dep_acct_pays", "pnc_dep_acct_receives", "unq_cpty_acct_id"]:
        hits.append(pay.where(norm(c) == target)
                       .groupBy(F.lit(c).alias("column"), F.col(c).alias("stored_value"))
                       .agg(F.count("*").alias("n_txn")))
    out = hits[0].unionByName(hits[1]).unionByName(hits[2]).limit(20).toPandas()
    return out

## 2 · Aggregation

Everything heavy happens in Spark on the cached ego rows; only small aggregates come back to
pandas. The one exception is a capped sample of transaction amounts for the histogram.
The totals of every aggregate are reconciled against each other before anything is shown.

In [ ]:
def _mode(df, key, col):
    """Most frequent non-null value of `col` per `key` (names vary: 'ACME INC' vs 'ACME, INC.')."""
    w = Window.partitionBy(key).orderBy(F.desc("_n"), F.col(col))
    return (df.where(F.col(col).isNotNull()).groupBy(key, col).agg(F.count("*").alias("_n"))
              .withColumn("_r", F.row_number().over(w)).where("_r = 1").select(key, col))


_PCTS = [0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
_PCT_NAMES = ["p10", "p25", "p50", "p75", "p90", "p99"]


def _with_pcts(pdf, col="pcts"):
    arr = np.vstack(pdf[col].apply(lambda v: list(v) if v is not None else [np.nan] * len(_PCTS)))
    for i, n in enumerate(_PCT_NAMES):
        pdf[n] = arr[:, i]
    return pdf.drop(columns=[col])


def ego_aggregates(e, sink=print):
    t0 = time.time()
    pct = F.expr(f"percentile_approx(amt, array({','.join(map(str, _PCTS))}), 10000)").alias("pcts")
    amount, n_txn = F.sum("amt").alias("amount"), F.count("*").alias("n_txn")
    n_cp = F.countDistinct("cp_key").alias("n_cp")

    qa = (e.groupBy("direction")
           .agg(n_txn, amount,
                F.sum(F.col("cp_key").isNull().cast("int")).alias("n_no_cp_id"),
                F.sum(F.when(F.col("cp_key").isNull(), F.col("amt")).otherwise(0.0)).alias("amt_no_cp_id"),
                F.sum("qa_acct_without_mdm").alias("n_acct_without_mdm"),
                F.sum((F.col("amt") <= 0).cast("int")).alias("n_nonpositive_amt"))
           .toPandas())

    prof = e.agg(F.collect_set("ego_name").alias("names"), F.collect_set("ego_mdm").alias("mdms"),
                 F.min("dt").alias("first_dt"), F.max("dt").alias("last_dt"),
                 F.countDistinct("month").alias("months_active")).first()

    ex = e.where(F.col("direction").isin("IN", "OUT") & F.col("cp_key").isNotNull())

    cp_dir = (ex.groupBy("direction", "cp_key")
                .agg(amount, n_txn, F.min("dt").alias("first_dt"), F.max("dt").alias("last_dt"),
                     F.countDistinct("month").alias("months_active"))
                .toPandas())
    cp_rail = ex.groupBy("direction", "cp_key", "rail").agg(amount, n_txn).toPandas()
    attr = (ex.groupBy("cp_key").agg(F.first("cp_type").alias("cp_type"),
                                     F.max("cp_acct").alias("cp_acct"), F.max("cp_mdm").alias("cp_mdm"))
              .join(_mode(ex, "cp_key", "cp_name"), "cp_key", "left")
              .join(_mode(ex, "cp_key", "cp_bank"), "cp_key", "left")
              .toPandas())
    dir_type = ex.groupBy("direction", "cp_type").agg(amount, n_txn, n_cp).toPandas()
    dir_stats = _with_pcts(ex.groupBy("direction").agg(amount, n_txn, n_cp, pct).toPandas())
    rail = _with_pcts(ex.groupBy("direction", "rail")
                        .agg(amount, n_txn, n_cp, F.max("amt").alias("max_txn"), pct).toPandas())
    rail_type = ex.groupBy("direction", "rail", "cp_type").agg(amount, n_txn, n_cp).toPandas()
    rail_cat = ex.groupBy("direction", "rail", "category").agg(amount, n_txn, n_cp).toPandas()
    month = ex.groupBy("direction", "month", "rail").agg(amount, n_txn).toPandas()
    month_cp = ex.groupBy("direction", "month", "cp_type").agg(amount, n_txn, n_cp).toPandas()
    bank = (ex.where(F.col("cp_type") == "external")
              .withColumn("bank", F.coalesce("cp_bank", F.lit("(bank not recorded)")))
              .groupBy("direction", "bank").agg(amount, n_txn, n_cp).toPandas())

    n_ex = int(dir_stats["n_txn"].sum()) if len(dir_stats) else 0
    frac = min(1.0, DIST_SAMPLE_MAX / max(n_ex, 1))
    sample = (ex.select("direction", "rail", "cp_type", "amt").sample(False, frac, seed=7).toPandas()
              if frac < 1 else ex.select("direction", "rail", "cp_type", "amt").toPandas())

    # ---- reconciliation: every aggregate must carry the same dollars ---------------------------
    ref = float(dir_stats["amount"].sum())
    for name, d in [("cp_dir", cp_dir), ("cp_rail", cp_rail), ("rail", rail), ("month", month),
                    ("rail_cat", rail_cat), ("dir_type", dir_type)]:
        got = float(d["amount"].sum())
        if abs(got - ref) > 1e-6 * max(abs(ref), 1.0):
            raise AssertionError(f"Reconciliation failed: {name} sums to {got:,.2f}, expected {ref:,.2f}")
    _log(f"aggregates built, reconciled at ${ref:,.0f} over {n_ex:,} txns", t0, sink)

    return dict(qa=qa, cp_dir=cp_dir, cp_rail=cp_rail, attr=attr, dir_type=dir_type,
                dir_stats=dir_stats, rail=rail, rail_type=rail_type, rail_cat=rail_cat,
                month=month, month_cp=month_cp, bank=bank, sample=sample, sample_frac=frac,
                ego_names=sorted(prof["names"] or []), ego_mdms=sorted(prof["mdms"] or []),
                first_dt=prof["first_dt"], last_dt=prof["last_dt"],
                months_active=prof["months_active"])


def alter_ties(pay, alters, ego_key, include_external, sink=print):
    """Payments between the ego's alters (the ego itself excluded).

    `alters` is a pandas frame of cp_key / cp_type / cp_acct / cp_mdm for the alters to search.
    Every observable tie has at least one internal end (external->external is not in the table),
    so the scan is pre-filtered on the internal alters' raw account columns before any key is
    built; the broadcast semi-joins on both ends then keep only alter<->alter rows."""
    t0 = time.time()
    internal = alters[alters["cp_type"] == "internal"]
    by_acct = internal.loc[~internal["cp_key"].str.startswith("PNC:MDM-"), "cp_acct"].dropna().unique().tolist()
    by_mdm = internal.loc[internal["cp_key"].str.startswith("PNC:MDM-"), "cp_mdm"].dropna().unique().tolist()
    if not by_acct and not by_mdm:
        _log("no internal alters -> no observable alter ties", sink=sink)
        return pd.DataFrame(columns=["src_key", "dst_key", "src_type", "dst_type", "rail",
                                     "amount", "n_txn", "first_dt", "last_dt"])
    cond = F.lit(False)
    if by_acct:
        cond = cond | F.col("pnc_dep_acct_pays").isin(by_acct) | F.col("pnc_dep_acct_receives").isin(by_acct)
    if by_mdm:
        cond = cond | F.col("mdm_id_pays").isin(by_mdm) | F.col("mdm_id_receives").isin(by_mdm)

    keys = spark.createDataFrame([(k,) for k in alters["cp_key"].astype(str).unique()], "k string")
    s = standardize(pay.where(cond))
    s = (s.where((F.col("src_key") != ego_key) & (F.col("dst_key") != ego_key)
                 & (F.col("src_key") != F.col("dst_key")))
          .join(F.broadcast(keys.withColumnRenamed("k", "src_key")), "src_key", "left_semi")
          .join(F.broadcast(keys.withColumnRenamed("k", "dst_key")), "dst_key", "left_semi"))
    if not include_external:
        s = s.where((F.col("src_type") == "internal") & (F.col("dst_type") == "internal"))
    ties = (s.groupBy("src_key", "dst_key", "src_type", "dst_type", "rail")
              .agg(F.sum("amt").alias("amount"), F.count("*").alias("n_txn"),
                   F.min("dt").alias("first_dt"), F.max("dt").alias("last_dt"))
              .toPandas())
    _log(f"alter ties: {ties.groupby(['src_key', 'dst_key']).ngroups if len(ties) else 0:,} "
         f"directed pairs among {len(alters):,} alters searched", t0, sink)
    return ties


def run_ego(acct, start=DEFAULT_START, end=DEFAULT_END, alter_cap=ALTER_CAP,
            include_external_ties=True, sink=print):
    """Full pipeline for one account. Returns a results dict consumed by every render_* function."""
    acct = str(acct).strip()
    if not acct:
        raise ValueError("Enter an account number.")
    t0 = time.time()
    _log(f"account {acct} | {start} → {end} | table {PAYMENTS_TABLE}", sink=sink)
    pay = load_payments(start, end, sink)
    e, ego_key = ego_transactions(pay, acct)
    e = e.persist()
    n = e.count()
    _log(f"{n:,} transactions touch the account", t0, sink)
    if n == 0:
        e.unpersist()
        cand = diagnose_missing(pay, acct)
        raise LookupError(
            f"No transactions for account '{acct}' in {start} → {end}.\n"
            + ("Near matches (whitespace / leading zeros stripped):\n" + cand.to_string(index=False)
               if len(cand) else "No near matches either: check the number and the date window."))
    try:
        R = ego_aggregates(e, sink)
    finally:
        e.unpersist()

    R.update(acct=acct, ego_key=ego_key, start=start, end=end, n_rows=n,
             include_external_ties=include_external_ties)
    if R["cp_dir"].empty:
        raise LookupError("The account has only self-transfers or unidentified counterparties in this window.")

    # alters ranked by total $ flow with the ego (both directions)
    flow = (R["cp_dir"].pivot_table(index="cp_key", columns="direction", values="amount",
                                    aggfunc="sum", fill_value=0.0)
                       .reindex(columns=["IN", "OUT"], fill_value=0.0))
    flow["total"] = flow["IN"] + flow["OUT"]
    R["flow"] = flow.sort_values("total", ascending=False)
    search = R["attr"].set_index("cp_key").loc[R["flow"].index[:alter_cap]].reset_index()
    R["alter_cap"], R["n_alters"], R["n_searched"] = alter_cap, len(R["flow"]), len(search)
    R["ties"] = alter_ties(pay, search, ego_key, include_external_ties, sink)

    safe = re.sub(r"[^0-9A-Za-z_-]", "_", acct)
    R["out_dir"] = os.path.join(OUT_ROOT, f"{safe}_{start}_{end}")
    os.makedirs(R["out_dir"], exist_ok=True)
    R["tables"] = build_tables(R)
    export_tables(R)
    _log(f"done → {R['out_dir']}", t0, sink)
    return R

## 3 · Summary tables (pandas, on the small aggregates)

In [ ]:
def _rail_mix(cp_rail, direction):
    d = cp_rail[cp_rail["direction"] == direction].copy()
    if d.empty:
        return pd.Series(dtype=object), pd.Series(dtype=object)
    tot = d.groupby("cp_key")["amount"].transform("sum")
    d["pct"] = np.where(tot > 0, d["amount"] / tot, np.nan)
    d = d.sort_values(["cp_key", "amount"], ascending=[True, False])
    d["s"] = d["rail"] + " " + (d["pct"] * 100).round(0).fillna(0).astype(int).astype(str) + "%"
    mix = d.groupby("cp_key").head(3).groupby("cp_key")["s"].agg(", ".join)
    dom = d.groupby("cp_key").head(1).set_index("cp_key")["rail"]
    return mix, dom


def counterparty_table(R, direction):
    """One row per account on one end of the ego's flow, ranked by dollars."""
    d = R["cp_dir"][R["cp_dir"]["direction"] == direction].merge(R["attr"], on="cp_key", how="left")
    if d.empty:
        return d
    mix, dom = _rail_mix(R["cp_rail"], direction)
    other = set(R["cp_dir"].loc[R["cp_dir"]["direction"] != direction, "cp_key"])
    d = d.sort_values("amount", ascending=False).reset_index(drop=True)
    tot = d["amount"].sum()
    d["share"] = d["amount"] / tot if tot else np.nan
    d["cum_share"] = d["share"].cumsum()
    d["avg_txn"] = d["amount"] / d["n_txn"]
    d["type"] = d["cp_type"].map(TYPE_LABEL)
    d["bank"] = np.where(d["cp_type"] == "internal", "PNC", d["cp_bank"].fillna("(not recorded)"))
    d["rail_mix"] = d["cp_key"].map(mix)
    d["dominant_rail"] = d["cp_key"].map(dom)
    d["same_customer"] = (d["cp_type"] == "internal") & d["cp_mdm"].isin(R["ego_mdms"])
    d["two_way"] = d["cp_key"].isin(other)
    d.index = np.arange(1, len(d) + 1)
    d.index.name = "rank"
    return d


def _conc(tbl):
    if tbl is None or tbl.empty:
        return dict(n_accounts=0, top1=np.nan, top5=np.nan, top10=np.nan, hhi=np.nan, effective_n=np.nan)
    s = tbl["share"].values
    hhi = float((s ** 2).sum())
    return dict(n_accounts=len(s), top1=s[:1].sum(), top5=s[:5].sum(), top10=s[:10].sum(),
                hhi=hhi, effective_n=1 / hhi if hhi else np.nan)


def build_tables(R):
    T = {}
    T["in"], T["out"] = counterparty_table(R, "IN"), counterparty_table(R, "OUT")

    # ---- profile & headline -------------------------------------------------------------------
    ds = R["dir_stats"].set_index("direction")
    dt = R["dir_type"]
    g = lambda d, col: float(ds[col].get(d, 0) or 0)
    ncp = lambda d, t: int(dt.loc[(dt["direction"] == d) & (dt["cp_type"] == t), "n_cp"].sum())
    head = []
    for d in ["IN", "OUT"]:
        head.append({"direction": DIR_LABEL[d], "amount": g(d, "amount"), "n_txn": int(g(d, "n_txn")),
                     "avg_txn": g(d, "amount") / g(d, "n_txn") if g(d, "n_txn") else np.nan,
                     "median_txn": g(d, "p50"), "p90_txn": g(d, "p90"),
                     "accounts": int(g(d, "n_cp")), "pnc_customer_accts": ncp(d, "internal"),
                     "external_accts": ncp(d, "external")})
    T["headline"] = pd.DataFrame(head).set_index("direction")
    both = set(T["in"]["cp_key"]) & set(T["out"]["cp_key"]) if len(T["in"]) and len(T["out"]) else set()
    net = g("IN", "amount") - g("OUT", "amount")
    T["profile"] = pd.DataFrame({"value": [
        R["acct"], "; ".join(map(str, R["ego_names"])) or "(no name)", "; ".join(map(str, R["ego_mdms"])),
        f"{R['start']} → {R['end']}", f"{R['first_dt']} → {R['last_dt']}", R["months_active"],
        f"{R['n_rows']:,}", f"{R['n_alters']:,}", f"{len(both):,}", f"${net:,.0f}"]},
        index=["Account", "Owner name(s)", "Owner MDM id(s)", "Window requested", "Activity observed",
               "Active months", "Transactions (all)", "Connected accounts (alters)",
               "Two-way accounts (send and receive)", "Net flow (in − out)"])

    # ---- QA -------------------------------------------------------------------------------------
    qa = R["qa"].set_index("direction")
    T["qa"] = qa

    # ---- internal vs external --------------------------------------------------------------------
    x = dt.copy()
    x["share_amt"] = x["amount"] / x.groupby("direction")["amount"].transform("sum")
    x["share_txn"] = x["n_txn"] / x.groupby("direction")["n_txn"].transform("sum")
    x["avg_txn"] = x["amount"] / x["n_txn"]
    x["direction"] = x["direction"].map(DIR_LABEL)
    x["cp_type"] = x["cp_type"].map(TYPE_LABEL)
    T["type_split"] = x.rename(columns={"cp_type": "counterparty type", "n_cp": "accounts"}) \
                       .set_index(["direction", "counterparty type"]).sort_index()

    # ---- rails -----------------------------------------------------------------------------------
    r = R["rail"].copy()
    r["share_amt"] = r["amount"] / r.groupby("direction")["amount"].transform("sum")
    r["share_txn"] = r["n_txn"] / r.groupby("direction")["n_txn"].transform("sum")
    r["avg_txn"] = r["amount"] / r["n_txn"]
    r["direction"] = r["direction"].map(DIR_LABEL)
    T["rails"] = (r.sort_values(["direction", "amount"], ascending=[True, False])
                   .set_index(["direction", "rail"])
                   [["amount", "share_amt", "n_txn", "share_txn", "avg_txn", "p10", "p50", "p90",
                     "p99", "max_txn", "n_cp"]].rename(columns={"n_cp": "accounts"}))

    rt = R["rail_type"].pivot_table(index=["direction", "rail"], columns="cp_type", values="amount",
                                    aggfunc="sum", fill_value=0.0)
    rt = rt.reindex(columns=["internal", "external"], fill_value=0.0)
    rt["external_share"] = rt["external"] / (rt["internal"] + rt["external"])
    rt = rt.rename(columns={"internal": "PNC customers $", "external": "External $"}).reset_index()
    rt["direction"] = rt["direction"].map(DIR_LABEL)
    T["rail_by_type"] = rt.set_index(["direction", "rail"]).sort_index()

    rc = R["rail_cat"].copy()
    rc["share_amt"] = rc["amount"] / rc.groupby("direction")["amount"].transform("sum")
    rc["direction"] = rc["direction"].map(DIR_LABEL)
    T["rail_category"] = (rc.sort_values(["direction", "amount"], ascending=[True, False])
                            .set_index(["direction", "rail", "category"])
                            .rename(columns={"n_cp": "accounts"}))

    # ---- concentration ----------------------------------------------------------------------------
    T["concentration"] = pd.DataFrame({DIR_LABEL["IN"]: _conc(T["in"]), DIR_LABEL["OUT"]: _conc(T["out"])}).T

    # ---- two-way accounts -------------------------------------------------------------------------
    if both:
        a = T["in"].set_index("cp_key")
        b = T["out"].set_index("cp_key")
        tw = pd.DataFrame({"name": a.loc[list(both), "cp_name"], "type": a.loc[list(both), "type"],
                           "account": a.loc[list(both), "cp_acct"], "bank": a.loc[list(both), "bank"],
                           "received_from": a.loc[list(both), "amount"], "sent_to": b.loc[list(both), "amount"],
                           "same_customer": a.loc[list(both), "same_customer"]})
        tw["net_to_account"] = tw["received_from"] - tw["sent_to"]
        tw["gross"] = tw["received_from"] + tw["sent_to"]
        T["two_way"] = tw.sort_values("gross", ascending=False).reset_index(drop=True)
    else:
        T["two_way"] = pd.DataFrame()

    # ---- banks ------------------------------------------------------------------------------------
    bk = R["bank"].copy()
    if len(bk):
        bk["share_of_external_amt"] = bk["amount"] / bk.groupby("direction")["amount"].transform("sum")
        bk["direction"] = bk["direction"].map(DIR_LABEL)
        T["banks"] = (bk.sort_values(["direction", "amount"], ascending=[True, False])
                        .set_index(["direction", "bank"]).rename(columns={"n_cp": "accounts"}))
    else:
        T["banks"] = pd.DataFrame()

    # ---- monthly ----------------------------------------------------------------------------------
    m = R["month_cp"].pivot_table(index="month", columns=["direction"], values="amount", aggfunc="sum",
                                  fill_value=0.0).reindex(columns=["IN", "OUT"], fill_value=0.0)
    mn = R["month_cp"].pivot_table(index="month", columns="direction", values="n_txn", aggfunc="sum",
                                   fill_value=0).reindex(columns=["IN", "OUT"], fill_value=0)
    mc = R["month_cp"].pivot_table(index="month", columns=["direction", "cp_type"], values="n_cp",
                                   aggfunc="sum", fill_value=0)
    monthly = pd.DataFrame({"received $": m["IN"], "sent $": m["OUT"], "net $": m["IN"] - m["OUT"],
                            "received txns": mn["IN"], "sent txns": mn["OUT"]})
    for d in ["IN", "OUT"]:
        for t in ["internal", "external"]:
            monthly[f"{'senders' if d == 'IN' else 'receivers'} ({'PNC' if t == 'internal' else 'ext'})"] = \
                mc[(d, t)] if (d, t) in mc.columns else 0
    T["monthly"] = monthly.sort_index()

    # ---- ties / embeddedness ----------------------------------------------------------------------
    ties = R["ties"]
    if len(ties):
        tp = ties.groupby(["src_key", "dst_key", "src_type", "dst_type"], as_index=False) \
                 .agg(amount=("amount", "sum"), n_txn=("n_txn", "sum"))
        tp["tie_kind"] = np.where((tp["src_type"] == "internal") & (tp["dst_type"] == "internal"),
                                  "PNC ↔ PNC", "PNC ↔ external")
        T["tie_kinds"] = tp.groupby("tie_kind").agg(directed_pairs=("amount", "size"),
                                                    amount=("amount", "sum"), n_txn=("n_txn", "sum"))
        deg = pd.concat([tp[["src_key", "dst_key", "amount"]].rename(columns={"src_key": "k", "dst_key": "o"}),
                         tp[["dst_key", "src_key", "amount"]].rename(columns={"dst_key": "k", "src_key": "o"})])
        emb = deg.groupby("k").agg(tie_partners=("o", "nunique"), tie_amount=("amount", "sum"))
        A = R["attr"].set_index("cp_key")
        emb["name"] = A.reindex(emb.index)["cp_name"]
        emb["type"] = A.reindex(emb.index)["cp_type"].map(TYPE_LABEL)
        emb["account"] = A.reindex(emb.index)["cp_acct"]
        emb["flow_with_account"] = R["flow"]["total"].reindex(emb.index)
        T["embedded"] = (emb.sort_values(["tie_partners", "tie_amount"], ascending=False)
                            .reset_index(drop=True)[["name", "type", "account", "tie_partners",
                                                     "tie_amount", "flow_with_account"]])
        T["tie_pairs"] = tp
    else:
        T["tie_kinds"] = T["embedded"] = T["tie_pairs"] = pd.DataFrame()
    return T


def export_tables(R):
    T = R["tables"]
    for k, v in T.items():
        if isinstance(v, pd.DataFrame) and len(v):
            v.to_csv(os.path.join(R["out_dir"], f"table_{k}.csv"))
    R["ties"].to_csv(os.path.join(R["out_dir"], "edges_alter_ties_by_rail.csv"), index=False)
    R["cp_rail"].to_csv(os.path.join(R["out_dir"], "edges_ego_by_rail.csv"), index=False)

## 4 · Display helpers and figures

In [ ]:
_MONEY = "${:,.0f}"
_PCT = "{:.1%}"
_INT = "{:,.0f}"


def show_table(df, title, money=(), pct=(), ints=(), note=None, max_rows=None):
    display(HTML(f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>"
                 + (f"<div style='color:#555;font-size:12px;margin-bottom:4px'>{note}</div>" if note else "")))
    if df is None or len(df) == 0:
        display(HTML("<i style='color:#777'>none</i>"))
        return
    d = df.head(max_rows) if max_rows else df
    fmt = {c: _MONEY for c in money if c in d.columns}
    fmt.update({c: _PCT for c in pct if c in d.columns})
    fmt.update({c: _INT for c in ints if c in d.columns})
    display(d.style.format(fmt, na_rep=""))


def _emit(R, fig, name):
    path = os.path.join(R["out_dir"], f"fig_{name}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if SHOW_FIGURES:
        display(fig)
    plt.close(fig)
    return path


_fmt_money_axis = FuncFormatter(lambda v, _: _money_short(v))


def fig_monthly(R):
    m = R["month"].pivot_table(index="month", columns=["direction", "rail"], values="amount",
                               aggfunc="sum", fill_value=0.0).sort_index()
    rails = sorted({r for _, r in m.columns}, key=lambda r: -R["rail"].loc[R["rail"]["rail"] == r, "amount"].sum())
    months = list(m.index)
    x = np.arange(len(months))
    fig, ax = plt.subplots(figsize=(13, 5.2))
    pos, neg = np.zeros(len(x)), np.zeros(len(x))
    for r in rails:
        vin = m[("IN", r)].values if ("IN", r) in m.columns else np.zeros(len(x))
        vout = m[("OUT", r)].values if ("OUT", r) in m.columns else np.zeros(len(x))
        ax.bar(x, vin, bottom=pos, color=rail_color(r), label=r, width=0.8)
        ax.bar(x, -vout, bottom=-neg, color=rail_color(r), width=0.8, alpha=0.75)
        pos += vin
        neg += vout
    ax.plot(x, pos - neg, color="black", lw=1.6, marker="o", ms=3.5, label="Net (in − out)")
    ax.axhline(0, color="#444", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(months, rotation=60, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(_fmt_money_axis)
    ax.set_title(f"Monthly flow by rail — received above the axis, sent below  ·  account {R['acct']}")
    ax.legend(ncol=min(len(rails) + 1, 7), fontsize=8, loc="upper left", frameon=False)
    ax.grid(axis="y", alpha=0.3)
    return fig


def fig_rail_mix(R):
    r = R["rail"]
    rails = r.groupby("rail")["amount"].sum().sort_values(ascending=False).index.tolist()
    rows = [("IN", "amount", "Received · $"), ("IN", "n_txn", "Received · # txns"),
            ("OUT", "amount", "Sent · $"), ("OUT", "n_txn", "Sent · # txns")]
    fig, ax = plt.subplots(figsize=(12, 3.6))
    for i, (d, col, lab) in enumerate(rows):
        sub = r[r["direction"] == d].set_index("rail")[col].reindex(rails).fillna(0)
        tot = sub.sum()
        left = 0.0
        for rail_name, v in sub.items():
            share = v / tot if tot else 0
            ax.barh(i, share, left=left, color=rail_color(rail_name),
                    label=rail_name if i == 0 else None, edgecolor="white", lw=0.5)
            if share >= 0.06:
                ax.text(left + share / 2, i, f"{share:.0%}", ha="center", va="center", fontsize=8, color="white")
            left += share
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([x[2] for x in rows])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_title("Rail mix — share of dollars vs share of transactions")
    ax.legend(ncol=min(len(rails), 8), fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
    return fig


def fig_top_counterparties(R, n=15):
    T = R["tables"]
    fig, axes = plt.subplots(1, 2, figsize=(15, max(4.5, 0.36 * n + 1.5)))
    for ax, key, title in [(axes[0], "in", "Top senders to the account"),
                           (axes[1], "out", "Top receivers from the account")]:
        t = T[key].head(n)
        if t.empty:
            ax.set_axis_off()
            ax.set_title(title + " — none")
            continue
        labels = [f"{_trunc(nm, 30)}{' ★' if sc else ''}" for nm, sc in zip(t["cp_name"].fillna(t["cp_acct"]), t["same_customer"])]
        y = np.arange(len(t))
        ax.barh(y, t["amount"], color=t["cp_type"].map(TYPE_COLOR))
        for yi, (v, s) in enumerate(zip(t["amount"], t["share"])):
            ax.text(v, yi, f"  {_money_short(v)} · {s:.0%}", va="center", fontsize=8)
        ax.set_yticks(y)
        ax.set_yticklabels(labels, fontsize=8)
        ax.invert_yaxis()
        ax.xaxis.set_major_formatter(_fmt_money_axis)
        ax.set_xlim(0, t["amount"].max() * 1.3)
        ax.set_title(title)
        ax.grid(axis="x", alpha=0.3)
    handles = [plt.Rectangle((0, 0), 1, 1, color=TYPE_COLOR[k]) for k in ["internal", "external"]]
    fig.legend(handles, [TYPE_LABEL["internal"], TYPE_LABEL["external"]], loc="lower center", ncol=2,
               frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.text(0.99, -0.02, "★ = account of the same customer as the ego", ha="right", fontsize=8, color="#555")
    fig.tight_layout()
    return fig


def fig_amount_distribution(R):
    s = R["sample"]
    s = s[s["amt"] > 0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=False)
    if s.empty:
        return fig
    bins = np.logspace(np.log10(max(s["amt"].min(), 0.01)), np.log10(s["amt"].max() * 1.01), 50)
    for ax, d in zip(axes, ["IN", "OUT"]):
        sd = s[s["direction"] == d]
        for r in sd.groupby("rail")["amt"].size().sort_values(ascending=False).index:
            ax.hist(sd.loc[sd["rail"] == r, "amt"], bins=bins, histtype="step", lw=1.6,
                    color=rail_color(r), label=f"{r} (n={int((sd['rail'] == r).sum() / R['sample_frac']):,})")
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(_fmt_money_axis)
        ax.set_title(f"Transaction size — {DIR_LABEL[d].lower()}")
        ax.set_xlabel("amount per transaction (log scale)")
        ax.legend(fontsize=8, frameon=False)
        ax.grid(alpha=0.3)
    if R["sample_frac"] < 1:
        fig.text(0.99, -0.03, f"random sample of {R['sample_frac']:.1%} of transactions; n scaled up",
                 ha="right", fontsize=8, color="#555")
    fig.tight_layout()
    return fig


def fig_concentration(R):
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    for key, d, c in [("in", "IN", "#1F5AA6"), ("out", "OUT", "#C0392B")]:
        t = R["tables"][key]
        if len(t):
            ax.plot(np.arange(1, len(t) + 1), t["cum_share"].values, color=c, lw=2,
                    label=f"{DIR_LABEL[d]} ({len(t):,} accounts)")
    ax.set_xscale("log")
    ax.set_ylim(0, 1.02)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.set_xlabel("counterparty rank (log)")
    ax.set_ylabel("cumulative share of dollars")
    ax.set_title("Concentration — how many accounts carry the dollars")
    ax.grid(alpha=0.3, which="both")
    ax.legend(frameon=False, fontsize=9)
    return fig


def fig_banks(R, n=12):
    bk = R["bank"]
    fig, axes = plt.subplots(1, 2, figsize=(14, max(3.5, 0.34 * n + 1.2)))
    for ax, d in zip(axes, ["IN", "OUT"]):
        t = bk[bk["direction"] == d].sort_values("amount", ascending=False).head(n)
        if t.empty:
            ax.set_axis_off()
            ax.set_title(f"External banks — {DIR_LABEL[d].lower()}: none")
            continue
        y = np.arange(len(t))
        ax.barh(y, t["amount"], color=C_EXT)
        for yi, (v, k) in enumerate(zip(t["amount"], t["n_cp"])):
            ax.text(v, yi, f"  {_money_short(v)} · {k:,} accts", va="center", fontsize=8)
        ax.set_yticks(y)
        ax.set_yticklabels([_trunc(b, 34) for b in t["bank"]], fontsize=8)
        ax.invert_yaxis()
        ax.xaxis.set_major_formatter(_fmt_money_axis)
        ax.set_xlim(0, t["amount"].max() * 1.35)
        ax.set_title(f"Counterparty banks — {DIR_LABEL[d].lower()}")
        ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    return fig


def fig_active_counterparties(R):
    mc = R["month_cp"].pivot_table(index="month", columns=["direction", "cp_type"], values="n_cp",
                                   aggfunc="sum", fill_value=0).sort_index()
    fig, ax = plt.subplots(figsize=(13, 4))
    styles = {("IN", "internal"): (C_INT, "-"), ("IN", "external"): (C_EXT, "-"),
              ("OUT", "internal"): (C_INT, "--"), ("OUT", "external"): (C_EXT, "--")}
    for (d, t), (c, ls) in styles.items():
        if (d, t) in mc.columns:
            ax.plot(mc.index, mc[(d, t)], color=c, ls=ls, lw=1.8, marker="o", ms=3,
                    label=f"{'senders' if d == 'IN' else 'receivers'} — {TYPE_LABEL[t]}")
    ax.set_title("Active counterparty accounts per month (solid = sending to the account, dashed = receiving from it)")
    ax.tick_params(axis="x", rotation=60, labelsize=8)
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8, ncol=2)
    return fig

## 5 · Network (pyvis / vis-network)

* **Nodes** — ★ the account (ego) · ● PNC customer account · ◆ external counterparty account.
  Light-orange fill marks an account of the **same customer** as the ego. Size ∝ √(dollars with the ego).
* **Edges** — arrows point payer → payee. Solid edges touch the ego; dashed edges are payments
  **between** connected accounts. Colour = dominant rail by dollars; width ∝ log(dollars).
* **Interaction** — drag, zoom, hover for details, click a node to highlight its neighbourhood,
  use the drop-down to find a node by name.

The HTML is self-contained (vis.js is inlined), so the saved file opens anywhere without network access.

In [ ]:
def _edge_title(src_name, dst_name, rails_df, first_dt=None, last_dt=None):
    tot, n = rails_df["amount"].sum(), int(rails_df["n_txn"].sum())
    lines = [f"{src_name}  →  {dst_name}", f"{_money_short(tot)} in {n:,} txns"]
    for _, r in rails_df.sort_values("amount", ascending=False).iterrows():
        lines.append(f"  {r['rail']}: {_money_short(r['amount'])} ({int(r['n_txn']):,})")
    if first_dt is not None:
        lines.append(f"{first_dt} → {last_dt}")
    return "\n".join(lines)


def build_network(R, max_nodes=MAX_VIS_NODES, show_ties=True, height="800px"):
    ego_key, A = R["ego_key"], R["attr"].set_index("cp_key")
    flow = R["flow"]
    vis = flow.head(max_nodes)
    keep = set(vis.index)
    cp_rail, cp_dir = R["cp_rail"], R["cp_dir"].set_index(["direction", "cp_key"])

    net = Network(height=height, width="100%", directed=True, notebook=False, cdn_resources="in_line",
                  bgcolor="#FFFFFF", font_color="#222222", neighborhood_highlight=True, select_menu=True)

    ego_name = R["ego_names"][0] if R["ego_names"] else "(account)"
    ego_tip = "\n".join([f"EGO ACCOUNT {R['acct']}", ego_name, f"MDM: {', '.join(map(str, R['ego_mdms']))}",
                         f"Received: {_money_short(flow['IN'].sum())} from {int((flow['IN'] > 0).sum()):,} accts",
                         f"Sent: {_money_short(flow['OUT'].sum())} to {int((flow['OUT'] > 0).sum()):,} accts"])
    net.add_node(ego_key, label=f"{_trunc(ego_name, 32)}\n{R['acct']}", title=ego_tip, shape="star", size=40,
                 color={"background": C_EGO, "border": "#8A3F06"}, x=0, y=0, fixed=True,
                 font={"size": 16, "face": "arial", "bold": True})

    fmax = max(float(vis["total"].max()), 1.0)
    mix_in, _ = _rail_mix(cp_rail, "IN")
    mix_out, _ = _rail_mix(cp_rail, "OUT")
    for k, row in vis.iterrows():
        a = A.loc[k]
        internal = a["cp_type"] == "internal"
        same = internal and a["cp_mdm"] in R["ego_mdms"]
        name = a["cp_name"] if isinstance(a["cp_name"], str) else f"(no name) {a['cp_acct']}"
        tip = [name, TYPE_LABEL[a["cp_type"]] + ("  — same customer as ego" if same else ""),
               f"Account: {a['cp_acct']}"]
        tip.append(f"MDM: {a['cp_mdm']}" if internal else f"Bank: {a['cp_bank'] if isinstance(a['cp_bank'], str) else '(not recorded)'}")
        if row["IN"] > 0:
            tip.append(f"Paid the account: {_money_short(row['IN'])}  [{mix_in.get(k, '')}]")
        if row["OUT"] > 0:
            tip.append(f"Received from the account: {_money_short(row['OUT'])}  [{mix_out.get(k, '')}]")
        bg = C_SIB if same else (C_INT if internal else C_EXT)
        border = C_EGO if same else ("#123A6E" if internal else "#1D6B48")
        net.add_node(k, label=_trunc(name, 26), title="\n".join(tip), shape="dot" if internal else "diamond",
                     size=float(10 + 28 * np.sqrt(row["total"] / fmax)),
                     color={"background": bg, "border": border, "highlight": {"background": bg, "border": "#000"}},
                     borderWidth=2.5 if same else 1.2, font={"size": 11, "face": "arial"})

    # ---- edges: ego <-> alter ---------------------------------------------------------------------
    edges = []
    cr = cp_rail[cp_rail["cp_key"].isin(keep)]
    for (d, k), grp in cr.groupby(["direction", "cp_key"]):
        nm = A.loc[k, "cp_name"] if isinstance(A.loc[k, "cp_name"], str) else A.loc[k, "cp_acct"]
        src, dst = (k, ego_key) if d == "IN" else (ego_key, k)
        sn, dn = (nm, ego_name) if d == "IN" else (ego_name, nm)
        cd = cp_dir.loc[(d, k)]
        edges.append(dict(src=src, dst=dst, amount=grp["amount"].sum(),
                          rail=grp.sort_values("amount", ascending=False)["rail"].iloc[0], tie=False,
                          title=_edge_title(sn, dn, grp, cd["first_dt"], cd["last_dt"])))

    # ---- edges: alter <-> alter -------------------------------------------------------------------
    n_ties_drawn = 0
    ties = R["ties"]
    if show_ties and len(ties):
        tv = ties[ties["src_key"].isin(keep) & ties["dst_key"].isin(keep)]
        for (s_, d_), grp in tv.groupby(["src_key", "dst_key"]):
            nm = lambda k: A.loc[k, "cp_name"] if isinstance(A.loc[k, "cp_name"], str) else A.loc[k, "cp_acct"]
            edges.append(dict(src=s_, dst=d_, amount=grp["amount"].sum(),
                              rail=grp.sort_values("amount", ascending=False)["rail"].iloc[0], tie=True,
                              title="Between connected accounts\n" + _edge_title(nm(s_), nm(d_), grp,
                                                                                  grp["first_dt"].min(), grp["last_dt"].max())))
            n_ties_drawn += 1

    if edges:
        la = np.log1p(np.array([max(e["amount"], 0) for e in edges]))
        lo, hi = la.min(), la.max()
        for e, v in zip(edges, la):
            w = 1.0 + 7.0 * ((v - lo) / (hi - lo) if hi > lo else 0.5)
            col = rail_color(e["rail"])
            net.add_edge(e["src"], e["dst"], title=e["title"], width=float(w * (0.7 if e["tie"] else 1.0)),
                         color={"color": col, "highlight": col, "opacity": 0.55 if e["tie"] else 0.9},
                         dashes=bool(e["tie"]))

    opts = {
        "edges": {"arrows": {"to": {"enabled": True, "scaleFactor": 0.55}}, "arrowStrikethrough": False,
                  # curvedCW bends A->B and B->A to opposite sides, so two-way pairs stay readable
                  "smooth": {"enabled": True, "type": "curvedCW", "roundness": 0.18}},
        "physics": {"solver": "forceAtlas2Based",
                    "forceAtlas2Based": {"gravitationalConstant": -90, "centralGravity": 0.012,
                                         "springLength": 170, "springConstant": 0.06, "avoidOverlap": 0.5},
                    "stabilization": {"enabled": True, "iterations": 300}},
        "interaction": {"hover": True, "navigationButtons": True, "multiselect": True, "tooltipDelay": 60},
    }
    net.set_options(json.dumps(opts))
    html_doc = net.generate_html(notebook=False)
    # the node-search drop-down lists raw ids (PNC:… / EXT:…); relabel with name · account · type
    for n in net.nodes:
        k = n["id"]
        if k == ego_key:
            lab = f"★ {ego_name} · {R['acct']} (ego)"
        else:
            a = A.loc[k]
            lab = f"{a['cp_name'] if isinstance(a['cp_name'], str) else '(no name)'} · {a['cp_acct']} · {TYPE_LABEL[a['cp_type']]}"
        html_doc = html_doc.replace(f'<option value="{k}">{k}</option>',
                                    f'<option value="{_html.escape(k)}">{_html.escape(lab)}</option>', 1)

    shown_amt = vis["total"].sum() / max(flow["total"].sum(), 1.0)
    rails_present = sorted({e["rail"] for e in edges})
    caption = (f"Showing {len(vis):,} of {len(flow):,} connected accounts ({shown_amt:.0%} of the account's dollar flow)"
               + (f" · {n_ties_drawn:,} payments-between-alters edges" if show_ties else " · alter ties hidden")
               + (f" · ties searched among top {R['n_searched']:,} alters" if R["n_searched"] < R["n_alters"] else ""))
    legend = _legend_html(rails_present, caption)
    # tooltips: keep line breaks (vis-network renders a plain-text title with white-space: nowrap)
    css = "<style>div.vis-tooltip{white-space:pre-line !important;font-family:arial;font-size:12px;max-width:520px}</style>"
    # the only non-inlined assets in the template are bootstrap CSS/JS (cosmetic); drop them so the
    # page never waits on a blocked CDN
    html_doc = re.sub(r"<(link|script)[^>]*bootstrap[^>]*>(\s*</script>)?", "", html_doc)
    html_doc = html_doc.replace("<body>", "<body>" + css + legend, 1)

    path = os.path.join(R["out_dir"], f"ego_network_{R['acct']}.html")
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_doc)
    return html_doc, path, caption


def _legend_html(rails, caption):
    sw = lambda c, shape: {
        "star": f"<span style='color:{c};font-size:17px'>★</span>",
        "dot": f"<span style='display:inline-block;width:11px;height:11px;border-radius:50%;background:{c}'></span>",
        "diamond": f"<span style='display:inline-block;width:10px;height:10px;background:{c};transform:rotate(45deg)'></span>",
    }[shape]
    item = lambda s, t: f"<span style='margin-right:16px;white-space:nowrap'>{s} {t}</span>"
    nodes = (item(sw(C_EGO, "star"), "account (ego)") + item(sw(C_INT, "dot"), "PNC customer account")
             + item(sw(C_EXT, "diamond"), "external counterparty account")
             + item(sw(C_SIB, "dot"), "same customer as ego"))
    edge_kind = (item("<span style='display:inline-block;width:26px;border-top:3px solid #555;vertical-align:middle'></span>", "payment with the ego")
                 + item("<span style='display:inline-block;width:26px;border-top:3px dashed #999;vertical-align:middle'></span>", "payment between connected accounts"))
    rail_items = "".join(item(f"<span style='display:inline-block;width:22px;height:4px;background:{rail_color(r)};vertical-align:middle'></span>", r) for r in rails)
    return ("<div style='font-family:arial;font-size:12px;color:#333;padding:6px 10px;border-bottom:1px solid #ddd'>"
            f"<div>{nodes}</div><div style='margin-top:4px'>{edge_kind} &nbsp; <b>edge colour = main rail:</b> {rail_items}</div>"
            f"<div style='margin-top:4px;color:#666'>{_html.escape(caption)} · arrows point payer → payee · "
            "click a node to isolate its neighbourhood</div></div>")


def show_network(html_doc, height=900):
    # srcdoc (not IFrame(src=path)) so the page renders regardless of where the notebook server
    # roots relative paths; IPython warns about raw <iframe> HTML, which is intended here.
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        display(HTML(f"<iframe srcdoc=\"{_html.escape(html_doc, quote=True)}\" "
                     f"style='width:100%;height:{height}px;border:1px solid #ddd'></iframe>"))

## 6 · Report sections (each renders into one tab of the app)

In [ ]:
def render_summary(R):
    T = R["tables"]
    show_table(T["profile"], "Account profile")
    show_table(T["headline"], "Both ends of the flow",
               money=["amount", "avg_txn", "median_txn", "p90_txn"],
               ints=["n_txn", "accounts", "pnc_customer_accts", "external_accts"],
               note="median / p90 are per-transaction amounts (approximate percentiles)")
    show_table(T["type_split"], "PNC customers vs external counterparties",
               money=["amount", "avg_txn"], pct=["share_amt", "share_txn"], ints=["n_txn", "accounts"])
    show_table(T["concentration"], "Concentration",
               pct=["top1", "top5", "top10"], ints=["n_accounts"],
               note="share of dollars carried by the top 1 / 5 / 10 accounts; effective_n = 1 / HHI "
                    "(the number of equal-sized counterparties that would give the same concentration)")
    _emit(R, fig_concentration(R), "concentration")
    qa = T["qa"].copy()
    note = ("SELF = the account paying itself (excluded from the network). "
            "n_no_cp_id = external side with no counterparty id (kept in totals above only if IN/OUT; "
            "excluded from the network). n_acct_without_mdm = a side with a PNC account but no MDM id, typed internal.")
    show_table(qa, "Data quality", money=["amount", "amt_no_cp_id"],
               ints=["n_txn", "n_no_cp_id", "n_acct_without_mdm", "n_nonpositive_amt"], note=note)


_CP_COLS = ["cp_name", "type", "cp_acct", "cp_mdm", "bank", "amount", "share", "cum_share", "n_txn",
            "avg_txn", "rail_mix", "months_active", "first_dt", "last_dt", "same_customer", "two_way"]


def render_counterparties(R):
    T = R["tables"]
    _emit(R, fig_top_counterparties(R), "top_counterparties")
    for key, title in [("in", "Senders — accounts that paid this account"),
                       ("out", "Receivers — accounts this account paid")]:
        t = T[key]
        show_table(t[_CP_COLS] if len(t) else t, f"{title} (top {TOP_N} of {len(t):,})",
                   money=["amount", "avg_txn"], pct=["share", "cum_share"], ints=["n_txn", "months_active"],
                   max_rows=TOP_N, note="full list in the CSV export")
    show_table(T["two_way"], "Two-way accounts — both sent to and received from this account",
               money=["received_from", "sent_to", "net_to_account", "gross"], max_rows=TOP_N,
               note="net_to_account > 0: the account is a net receiver from this counterparty")


def render_rails(R):
    T = R["tables"]
    _emit(R, fig_rail_mix(R), "rail_mix")
    show_table(T["rails"], "Payment rails by direction",
               money=["amount", "avg_txn", "p10", "p50", "p90", "p99", "max_txn"],
               pct=["share_amt", "share_txn"], ints=["n_txn", "accounts"],
               note="p10 … p99: per-transaction amount percentiles (approximate)")
    show_table(T["rail_by_type"], "Rail × counterparty type (dollars)",
               money=["PNC customers $", "External $"], pct=["external_share"])
    show_table(T["rail_category"], "Rail × category", money=["amount"], pct=["share_amt"],
               ints=["n_txn", "accounts"])
    _emit(R, fig_amount_distribution(R), "amount_distribution")


def render_time(R):
    _emit(R, fig_monthly(R), "monthly_flow")
    _emit(R, fig_active_counterparties(R), "active_counterparties")
    show_table(R["tables"]["monthly"], "Monthly flow and active counterparties",
               money=["received $", "sent $", "net $"],
               ints=[c for c in R["tables"]["monthly"].columns if "$" not in c])


def render_banks(R):
    _emit(R, fig_banks(R), "external_banks")
    show_table(R["tables"]["banks"], "External counterparties by their bank",
               money=["amount"], pct=["share_of_external_amt"], ints=["n_txn", "accounts"])


def render_ties(R):
    T = R["tables"]
    note = (f"Searched the top {R['n_searched']:,} of {R['n_alters']:,} alters by dollar flow with the account"
            + ("" if R["include_external_ties"] else "; PNC ↔ PNC ties only")
            + ". External → external payments are not in the source table, so those ties are never observable.")
    show_table(T["tie_kinds"], "Payments between the account's connected accounts",
               money=["amount"], ints=["directed_pairs", "n_txn"], note=note)
    show_table(T["embedded"], "Most embedded connected accounts (most tie partners inside the ego network)",
               money=["tie_amount", "flow_with_account"], ints=["tie_partners"], max_rows=TOP_N)

## 7 · The app

Type the account number, pick the window, press **Build**. *Redraw network* re-renders the picture
(node count, ties on/off) from the cached results without touching Spark.

Programmatic use, without the widgets:
```python
R = run_ego("1234567890", "2025-01-01", "2025-12-31")
html_doc, path, caption = build_network(R); show_network(html_doc)
R["tables"]["in"].head(20)
```

In [ ]:
class EgoApp:
    TABS = ["Network", "Summary", "Senders & receivers", "Rails", "Over time", "External banks",
            "Ties between alters", "Log"]

    def __init__(self):
        lay = lambda w: W.Layout(width=w)
        st = {"description_width": "initial"}
        self.acct = W.Text(description="PNC account #", placeholder="deposit account number", layout=lay("330px"), style=st)
        self.start = W.DatePicker(description="From", value=date.fromisoformat(DEFAULT_START), style=st)
        self.end = W.DatePicker(description="To", value=date.fromisoformat(DEFAULT_END), style=st)
        self.cap = W.BoundedIntText(description="Alters searched for ties", value=ALTER_CAP, min=0,
                                    max=100_000, step=100, layout=lay("250px"), style=st)
        self.ext = W.Checkbox(description="include ties to external counterparties", value=True, indent=False)
        self.nvis = W.IntSlider(description="Nodes drawn", value=MAX_VIS_NODES, min=10, max=600, step=10,
                                continuous_update=False, layout=lay("380px"), style=st)
        self.ties = W.Checkbox(description="show ties between connected accounts", value=True, indent=False)
        self.build = W.Button(description="Build", button_style="primary", icon="play", layout=lay("110px"))
        self.redraw = W.Button(description="Redraw network", icon="refresh", layout=lay("160px"), disabled=True)
        self.status = W.HTML()
        self.outs = [W.Output() for _ in self.TABS]
        self.tabs = W.Tab(children=self.outs)
        for i, t in enumerate(self.TABS):
            self.tabs.set_title(i, t)
        self.R = None
        self.build.on_click(self._on_build)
        self.redraw.on_click(self._on_redraw)
        self.ui = W.VBox([
            W.HTML("<b style='font-size:15px'>PKG · account ego network</b>"),
            W.HBox([self.acct, self.start, self.end, self.build]),
            W.HBox([self.cap, self.ext]),
            W.HBox([self.nvis, self.ties, self.redraw]),
            self.status, self.tabs])

    def _set_status(self, msg, color="#333"):
        self.status.value = f"<span style='color:{color}'>{_html.escape(msg)}</span>"

    def _log(self, msg):
        with self.outs[-1]:
            print(msg)

    def _draw(self):
        with self.outs[0]:
            self.outs[0].clear_output(wait=True)
            html_doc, path, caption = build_network(self.R, max_nodes=self.nvis.value, show_ties=self.ties.value)
            show_network(html_doc)
            print(f"saved: {path}")

    def _on_redraw(self, _):
        if self.R is not None:
            self._draw()

    def _on_build(self, _):
        acct = self.acct.value.strip()
        if not acct:
            self._set_status("Enter an account number.", "#B00020")
            return
        if self.start.value is None or self.end.value is None or self.start.value > self.end.value:
            self._set_status("Pick a valid date window.", "#B00020")
            return
        for o in self.outs:
            o.clear_output()
        self.build.disabled = self.redraw.disabled = True
        self._set_status(f"Querying {PAYMENTS_TABLE} for {acct} … (see the Log tab)")
        t0 = time.time()
        try:
            self.R = run_ego(acct, self.start.value.isoformat(), self.end.value.isoformat(),
                             alter_cap=int(self.cap.value), include_external_ties=self.ext.value,
                             sink=self._log)
            self._draw()
            for out, fn in zip(self.outs[1:-1], [render_summary, render_counterparties, render_rails,
                                                  render_time, render_banks, render_ties]):
                with out:
                    fn(self.R)
            R = self.R
            self._set_status(f"Account {acct}: {R['n_rows']:,} transactions, {R['n_alters']:,} connected accounts "
                             f"· {time.time() - t0:.0f}s · outputs in {R['out_dir']}", "#1B5E20")
            self.redraw.disabled = False
        except Exception as ex:  # surface the error in the app, full traceback in the Log tab
            self._set_status(f"{type(ex).__name__}: {ex}".split("\n")[0], "#B00020")
            with self.outs[-1]:
                import traceback
                traceback.print_exc()
            self.tabs.selected_index = len(self.TABS) - 1
        finally:
            self.build.disabled = False


app = EgoApp()
display(app.ui)